# Phase 4: Beat the Paper's 0.65

## Where we stand

Phase 3 baseline: VGG-Face + tuned SVR achieved **r = 0.626** overall on the test set. Paper's number is 0.65. Gap to close: 0.024.

## What we'll try (each independent — stop when we beat 0.65)

- **A:** Concatenated VGG-Face + FaceNet features with tuned SVR
- **B:** Ensemble (average) of VGG-Face SVR and FaceNet SVR predictions
- **C:** Other regression models on the best feature set (Ridge, Gradient Boosting, Random Forest)
- **D:** Wider SVR hyperparameter grid on the best feature set
- **E:** Different feature scalers

## What we already know going in

- VGG-Face SVR (tuned): r = 0.626, best params `C=10, gamma='scale', epsilon=0.1`
- FaceNet SVR (tuned): r = 0.614, best params `C=1, gamma='scale', epsilon=1.0`

## Note on time

Some experiments use GridSearchCV which can take 20-60 minutes each. We'll use moderate grids and the tqdm helper to keep things reasonable.

## Cell 1: Setup, helpers, load features

Same setup as Phase 3 plus the `tqdm_joblib` helper for progress bars on GridSearchCV.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import contextlib, time, os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV

PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final'
FEATURES_DIR = f'{PROJECT_ROOT}/features'
MODELS_DIR   = f'{PROJECT_ROOT}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

# tqdm-aware GridSearchCV helper (same one I gave you earlier)
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

def pearson_r(y_true, y_pred):
    return pearsonr(y_true, y_pred)[0]
pearson_scorer = make_scorer(pearson_r, greater_is_better=True)

def load_features(tag):
    return {
        'X_train':      np.load(f'{FEATURES_DIR}/X_train_{tag}.npy'),
        'X_test':       np.load(f'{FEATURES_DIR}/X_test_{tag}.npy'),
        'y_train':      np.load(f'{FEATURES_DIR}/y_train_{tag}.npy'),
        'y_test':       np.load(f'{FEATURES_DIR}/y_test_{tag}.npy'),
        'gender_train': np.load(f'{FEATURES_DIR}/gender_train_{tag}.npy'),
        'gender_test':  np.load(f'{FEATURES_DIR}/gender_test_{tag}.npy'),
    }

def evaluate(y_true, y_pred, gender, label='model'):
    r_o = pearsonr(y_true, y_pred)[0]
    r_m = pearsonr(y_true[gender=='Male'],   y_pred[gender=='Male'])[0]
    r_f = pearsonr(y_true[gender=='Female'], y_pred[gender=='Female'])[0]
    mae = mean_absolute_error(y_true, y_pred)
    print(f'\n=== {label} ===')
    print(f'  Overall r: {r_o:.4f}    Male r: {r_m:.4f}    Female r: {r_f:.4f}    MAE: {mae:.2f}')
    if r_o >= 0.65:
        print(f'  *** BEAT THE PAPER (target 0.65) ***')
    return {'label': label, 'r_overall': r_o, 'r_male': r_m, 'r_female': r_f, 'mae': mae}

vgg = load_features('VGG_Face')
fn  = load_features('Facenet512')

print(f'VGG-Face:  X_train {vgg["X_train"].shape}, X_test {vgg["X_test"].shape}')
print(f'FaceNet:   X_train {fn["X_train"].shape},  X_test {fn["X_test"].shape}')

# Sanity: same labels in both
assert np.allclose(vgg['y_train'], fn['y_train'])
assert np.allclose(vgg['y_test'],  fn['y_test'])
y_train = vgg['y_train']
y_test  = vgg['y_test']
gender_test = vgg['gender_test']
print('Labels match across feature sets. Good.')

# Track all results so we can compare at the end
all_experiments = []

Mounted at /content/drive
VGG-Face:  X_train (3123, 4096), X_test (741, 4096)
FaceNet:   X_train (3123, 512),  X_test (741, 512)
Labels match across feature sets. Good.


## Cell 2: Load Phase 3 baseline numbers as reference

We reload the saved tuned SVR models so we can see the baseline numbers in our final comparison without retraining.

In [2]:
vgg_baseline_model = joblib.load(f'{MODELS_DIR}/svr_vgg_tuned.joblib')
fn_baseline_model  = joblib.load(f'{MODELS_DIR}/svr_facenet_tuned.joblib')

vgg_pred_baseline = vgg_baseline_model.predict(vgg['X_test'])
fn_pred_baseline  = fn_baseline_model.predict(fn['X_test'])

all_experiments.append(evaluate(y_test, vgg_pred_baseline, gender_test, 'Baseline: VGG-Face SVR (tuned)'))
all_experiments.append(evaluate(y_test, fn_pred_baseline,  gender_test, 'Baseline: FaceNet SVR (tuned)'))


=== Baseline: VGG-Face SVR (tuned) ===
  Overall r: 0.6261    Male r: 0.6233    Female r: 0.6353    MAE: 5.15

=== Baseline: FaceNet SVR (tuned) ===
  Overall r: 0.6144    Male r: 0.6396    Female r: 0.5878    MAE: 5.32


## Experiment A: Concatenated features

Stack VGG-Face (4,096-dim) and FaceNet (512-dim) embeddings side-by-side per image to get 4,608-dim features. Then run the same GridSearchCV.

**Why this might help:** the two models were trained differently and capture overlapping but distinct face information. Combining them gives the SVR access to both views.

**Why it might not:** higher dimensions can hurt with limited data, and StandardScaler treats each dimension independently so the two feature blocks may end up at compatible scales already (or not — we'll see).

In [3]:
# Stack along the feature axis (axis=1)
X_train_concat = np.hstack([vgg['X_train'], fn['X_train']])
X_test_concat  = np.hstack([vgg['X_test'],  fn['X_test']])
print(f'Concatenated shapes: train {X_train_concat.shape}, test {X_test_concat.shape}')

param_grid = {
    'svr__C':       [1, 10, 100],
    'svr__gamma':   ['scale', 0.001, 0.01, 0.1],
    'svr__epsilon': [0.1, 0.5, 1.0],
}
n_fits = 1
for v in param_grid.values():
    n_fits *= len(v)
n_fits *= 5

pipe = Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='rbf'))])

print(f'\nTuning concat-SVR: {n_fits} fits  (~80 min on a slow CPU, less on faster ones)')
t0 = time.time()
with tqdm_joblib(tqdm(desc='Concat GridSearch', total=n_fits)):
    grid = GridSearchCV(pipe, param_grid=param_grid, scoring=pearson_scorer,
                       cv=5, n_jobs=-1, verbose=0)
    grid.fit(X_train_concat, y_train)
print(f'\nDone in {(time.time()-t0)/60:.1f} min')
print(f'Best CV r: {grid.best_score_:.4f}')
print(f'Best params: {grid.best_params_}')

concat_model = grid.best_estimator_
y_pred_concat = concat_model.predict(X_test_concat)
all_experiments.append(evaluate(y_test, y_pred_concat, gender_test, 'A: Concat VGG+FaceNet + SVR'))

# Save in case it's the winner
joblib.dump(concat_model, f'{MODELS_DIR}/svr_concat_tuned.joblib')

Concatenated shapes: train (3123, 4608), test (741, 4608)

Tuning concat-SVR: 180 fits  (~80 min on a slow CPU, less on faster ones)


Concat GridSearch:   0%|          | 0/180 [00:00<?, ?it/s]


Done in 72.2 min
Best CV r: 0.6681
Best params: {'svr__C': 10, 'svr__epsilon': 0.1, 'svr__gamma': 'scale'}

=== A: Concat VGG+FaceNet + SVR ===
  Overall r: 0.6436    Male r: 0.6499    Female r: 0.6415    MAE: 5.07


['/content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final/models/svr_concat_tuned.joblib']

## Experiment B: Ensemble of VGG-Face SVR and FaceNet SVR

Simplest ensemble: average the two models' predictions.

**Why this might help:** the two models make different mistakes. Averaging often reduces total error because random errors cancel out while consistent signal stays.

**Why it might not:** if both models are wrong in similar ways (e.g., both regress to mean for high-BMI faces), averaging won't fix it.

In [4]:
# Average the two baseline models' predictions
y_pred_ensemble = (vgg_pred_baseline + fn_pred_baseline) / 2
all_experiments.append(evaluate(y_test, y_pred_ensemble, gender_test, 'B: Ensemble (avg VGG + FaceNet)'))

# Also try weighted average favoring the better baseline (VGG)
y_pred_weighted = 0.7 * vgg_pred_baseline + 0.3 * fn_pred_baseline
all_experiments.append(evaluate(y_test, y_pred_weighted, gender_test, 'B2: Weighted ensemble (0.7 VGG + 0.3 FN)'))


=== B: Ensemble (avg VGG + FaceNet) ===
  Overall r: 0.6469    Male r: 0.6555    Female r: 0.6427    MAE: 5.11

=== B2: Weighted ensemble (0.7 VGG + 0.3 FN) ===
  Overall r: 0.6435    Male r: 0.6471    Female r: 0.6455    MAE: 5.10


## Experiment C: Other regression models on the best feature set

We try Ridge, Gradient Boosting, and Random Forest. These are quick to train so we'll run them on whichever features performed best so far (default to concatenated since A is most promising).

**Quick intuitions:**
- **Ridge** is linear regression with L2 regularization. Fast, usually a weaker baseline but sometimes surprises us when features are very informative.
- **Gradient Boosting** builds an ensemble of small decision trees, each one correcting the previous one's errors. Often strong on tabular data.
- **Random Forest** averages many independent decision trees. Usually a step down from gradient boosting on most regression tasks but quick to try.

In [5]:
# Pick the best features so far (will be concat if A worked, else fall back to VGG-Face alone)
best_so_far = max(all_experiments, key=lambda r: r['r_overall'])
print(f'Best so far: {best_so_far["label"]} (r={best_so_far["r_overall"]:.4f})')

if 'Concat' in best_so_far['label']:
    X_tr, X_te = X_train_concat, X_test_concat
    feat_label = 'concat'
else:
    X_tr, X_te = vgg['X_train'], vgg['X_test']
    feat_label = 'VGG-Face'
print(f'Using {feat_label} features for Experiment C')

# Ridge
print('\nRidge regression...')
ridge_pipe = Pipeline([('scaler', StandardScaler()), ('reg', Ridge())])
ridge_grid = GridSearchCV(ridge_pipe, {'reg__alpha': [0.1, 1.0, 10.0, 100.0]},
                          scoring=pearson_scorer, cv=5, n_jobs=-1)
ridge_grid.fit(X_tr, y_train)
y_pred_ridge = ridge_grid.best_estimator_.predict(X_te)
print(f'  best alpha: {ridge_grid.best_params_}')
all_experiments.append(evaluate(y_test, y_pred_ridge, gender_test, f'C1: Ridge on {feat_label}'))

# Gradient Boosting
print('\nGradient Boosting (this one takes a few minutes)...')
t0 = time.time()
gb_pipe = Pipeline([('scaler', StandardScaler()),
                    ('reg', GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42))])
gb_pipe.fit(X_tr, y_train)
y_pred_gb = gb_pipe.predict(X_te)
print(f'  fit in {time.time()-t0:.1f}s')
all_experiments.append(evaluate(y_test, y_pred_gb, gender_test, f'C2: Gradient Boosting on {feat_label}'))

# Random Forest
print('\nRandom Forest...')
t0 = time.time()
rf_pipe = Pipeline([('scaler', StandardScaler()),
                    ('reg', RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42))])
rf_pipe.fit(X_tr, y_train)
y_pred_rf = rf_pipe.predict(X_te)
print(f'  fit in {time.time()-t0:.1f}s')
all_experiments.append(evaluate(y_test, y_pred_rf, gender_test, f'C3: Random Forest on {feat_label}'))

Best so far: B: Ensemble (avg VGG + FaceNet) (r=0.6469)
Using VGG-Face features for Experiment C

Ridge regression...
  best alpha: {'reg__alpha': 100.0}

=== C1: Ridge on VGG-Face ===
  Overall r: 0.4121    Male r: 0.3961    Female r: 0.4474    MAE: 7.73

Gradient Boosting (this one takes a few minutes)...
  fit in 228.5s

=== C2: Gradient Boosting on VGG-Face ===
  Overall r: 0.6271    Male r: 0.6245    Female r: 0.6328    MAE: 5.12

Random Forest...
  fit in 528.5s

=== C3: Random Forest on VGG-Face ===
  Overall r: 0.6215    Male r: 0.6300    Female r: 0.6140    MAE: 5.24


## Checkpoint: where do we stand?

Let's see if we already beat 0.65 before running the more time-consuming experiments. If the answer is yes, you can skip Experiments D and E and go straight to Cell 11 to save the winner.

In [6]:
checkpoint_df = pd.DataFrame(all_experiments)[['label', 'r_overall', 'r_male', 'r_female', 'mae']]
checkpoint_df = checkpoint_df.sort_values('r_overall', ascending=False).reset_index(drop=True)
print('Results so far (sorted by overall r):')
print(checkpoint_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

best_now = checkpoint_df.iloc[0]
print(f'\nCurrent best: {best_now["label"]} with r = {best_now["r_overall"]:.4f}')
if best_now['r_overall'] >= 0.65:
    print('*** Already beating 0.65! You can skip to Cell 11 if you want. ***')
else:
    print(f'Gap to 0.65: {0.65 - best_now["r_overall"]:.4f}. Continue with D and E to push further.')

Results so far (sorted by overall r):
                                   label  r_overall  r_male  r_female    mae
         B: Ensemble (avg VGG + FaceNet)     0.6469  0.6555    0.6427 5.1134
             A: Concat VGG+FaceNet + SVR     0.6436  0.6499    0.6415 5.0668
B2: Weighted ensemble (0.7 VGG + 0.3 FN)     0.6435  0.6471    0.6455 5.0976
       C2: Gradient Boosting on VGG-Face     0.6271  0.6245    0.6328 5.1222
          Baseline: VGG-Face SVR (tuned)     0.6261  0.6233    0.6353 5.1487
           C3: Random Forest on VGG-Face     0.6215  0.6300    0.6140 5.2437
           Baseline: FaceNet SVR (tuned)     0.6144  0.6396    0.5878 5.3189
                   C1: Ridge on VGG-Face     0.4121  0.3961    0.4474 7.7285

Current best: B: Ensemble (avg VGG + FaceNet) with r = 0.6469
Gap to 0.65: 0.0031. Continue with D and E to push further.


## Experiment D: Wider SVR grid on the best feature set

Phase 3's grid topped out at C=100. Best param was C=10, which is well inside the grid. But the gamma options (`'scale'`, 0.001, 0.01, 0.1) were sparse. Let's expand the grid around the winners.

**Caveat:** wider grids take longer. This cell could take 30-60 minutes. If we already beat 0.65 in the checkpoint above, skip it.

In [7]:
# Use whichever features performed best in concat or baseline
best_so_far = max(all_experiments, key=lambda r: r['r_overall'])
if 'Concat' in best_so_far['label']:
    X_tr, X_te = X_train_concat, X_test_concat
    feat_label = 'concat'
else:
    X_tr, X_te = vgg['X_train'], vgg['X_test']
    feat_label = 'VGG-Face'
print(f'Running wider SVR grid on {feat_label} features')

wider_grid = {
    'svr__C':       [5, 10, 20, 50],
    'svr__gamma':   ['scale', 0.0001, 0.0005, 0.001, 0.005],
    'svr__epsilon': [0.05, 0.1, 0.2, 0.5],
}
n_fits = 1
for v in wider_grid.values():
    n_fits *= len(v)
n_fits *= 5
print(f'Total fits: {n_fits}')

pipe = Pipeline([('scaler', StandardScaler()), ('svr', SVR(kernel='rbf'))])
t0 = time.time()
with tqdm_joblib(tqdm(desc='Wider SVR grid', total=n_fits)):
    wide_grid = GridSearchCV(pipe, wider_grid, scoring=pearson_scorer,
                             cv=5, n_jobs=-1, verbose=0)
    wide_grid.fit(X_tr, y_train)
print(f'\nDone in {(time.time()-t0)/60:.1f} min')
print(f'Best CV r: {wide_grid.best_score_:.4f}')
print(f'Best params: {wide_grid.best_params_}')

y_pred_wider = wide_grid.best_estimator_.predict(X_te)
all_experiments.append(evaluate(y_test, y_pred_wider, gender_test, f'D: Wider SVR grid on {feat_label}'))

joblib.dump(wide_grid.best_estimator_, f'{MODELS_DIR}/svr_wider_{feat_label}.joblib')

Running wider SVR grid on VGG-Face features
Total fits: 400


Wider SVR grid:   0%|          | 0/400 [00:00<?, ?it/s]


Done in 141.6 min
Best CV r: 0.6514
Best params: {'svr__C': 10, 'svr__epsilon': 0.05, 'svr__gamma': 'scale'}

=== D: Wider SVR grid on VGG-Face ===
  Overall r: 0.6260    Male r: 0.6231    Female r: 0.6352    MAE: 5.15


['/content/drive/MyDrive/Colab Notebooks/uChicago/ML/ML Final/models/svr_wider_VGG-Face.joblib']

## Experiment E: Different scalers

We've used StandardScaler everywhere. Try MinMaxScaler (each feature scaled to [0, 1]) and Normalizer (each sample's vector scaled to unit length). Quick to try.

We use the wider grid's best SVR params and just swap the scaler.

In [8]:
# Use best-so-far feature set and best-so-far SVR params
best_now = max(all_experiments, key=lambda r: r['r_overall'])
if 'Concat' in best_now['label'] or 'concat' in best_now['label']:
    X_tr, X_te = X_train_concat, X_test_concat
    feat_label = 'concat'
else:
    X_tr, X_te = vgg['X_train'], vgg['X_test']
    feat_label = 'VGG-Face'

# Use whatever SVR params we know best (from wider grid if it ran, else original tuned)
if 'wide_grid' in dir():
    svr_params = wide_grid.best_params_
else:
    svr_params = {'svr__C': 10, 'svr__gamma': 'scale', 'svr__epsilon': 0.1}

# Strip the 'svr__' prefix
svr_kwargs = {k.replace('svr__', ''): v for k, v in svr_params.items()}
print(f'Using features: {feat_label}, SVR params: {svr_kwargs}')

for scaler_name, scaler in [('MinMax', MinMaxScaler()), ('Normalizer', Normalizer())]:
    pipe = Pipeline([('scaler', scaler), ('svr', SVR(kernel='rbf', **svr_kwargs))])
    pipe.fit(X_tr, y_train)
    y_pred = pipe.predict(X_te)
    all_experiments.append(evaluate(y_test, y_pred, gender_test, f'E: {scaler_name} scaler on {feat_label}'))

Using features: VGG-Face, SVR params: {'C': 10, 'epsilon': 0.05, 'gamma': 'scale'}

=== E: MinMax scaler on VGG-Face ===
  Overall r: 0.6339    Male r: 0.6329    Female r: 0.6411    MAE: 5.08

=== E: Normalizer scaler on VGG-Face ===
  Overall r: 0.6353    Male r: 0.6367    Female r: 0.6382    MAE: 5.06


## Cell 11: Final comparison and pick the winner

Compile every experiment into one table sorted by overall Pearson r. Highlight which ones beat the paper's 0.65.

In [9]:
final_df = pd.DataFrame(all_experiments)[['label', 'r_overall', 'r_male', 'r_female', 'mae']]
final_df['beats_paper'] = final_df['r_overall'] >= 0.65
final_df = final_df.sort_values('r_overall', ascending=False).reset_index(drop=True)

# Add paper reference row
paper_row = pd.DataFrame([{
    'label': 'PAPER (target)', 'r_overall': 0.65, 'r_male': 0.71,
    'r_female': 0.57, 'mae': float('nan'), 'beats_paper': False,
}])
final_df = pd.concat([final_df, paper_row], ignore_index=True).sort_values('r_overall', ascending=False).reset_index(drop=True)

print('=== FINAL COMPARISON ===')
print(final_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

winner = final_df[final_df['label'] != 'PAPER (target)'].iloc[0]
print(f'\nOverall winner: {winner["label"]}')
print(f'  r = {winner["r_overall"]:.4f}')
if winner['r_overall'] >= 0.65:
    print(f'  *** BEATS THE PAPER (0.65) by {winner["r_overall"] - 0.65:.4f} ***')
else:
    print(f'  Falls short of 0.65 by {0.65 - winner["r_overall"]:.4f}')

# Save the final results table for the write-up
final_df.to_csv(f'{MODELS_DIR}/phase4_final_results.csv', index=False)
print(f'\nSaved final results table to {MODELS_DIR}/phase4_final_results.csv')

=== FINAL COMPARISON ===
                                   label  r_overall  r_male  r_female    mae  beats_paper
                          PAPER (target)     0.6500  0.7100    0.5700    NaN        False
         B: Ensemble (avg VGG + FaceNet)     0.6469  0.6555    0.6427 5.1134        False
             A: Concat VGG+FaceNet + SVR     0.6436  0.6499    0.6415 5.0668        False
B2: Weighted ensemble (0.7 VGG + 0.3 FN)     0.6435  0.6471    0.6455 5.0976        False
        E: Normalizer scaler on VGG-Face     0.6353  0.6367    0.6382 5.0566        False
            E: MinMax scaler on VGG-Face     0.6339  0.6329    0.6411 5.0834        False
       C2: Gradient Boosting on VGG-Face     0.6271  0.6245    0.6328 5.1222        False
          Baseline: VGG-Face SVR (tuned)     0.6261  0.6233    0.6353 5.1487        False
           D: Wider SVR grid on VGG-Face     0.6260  0.6231    0.6352 5.1495        False
           C3: Random Forest on VGG-Face     0.6215  0.6300    0.6140 5.243

## Cell 12: Document the winner for Phase 5 (demo)

This cell saves a small JSON file documenting which model won and what feature extraction it needs. Phase 5 will read this to set up the demo.

In [10]:
import json

winner_info = {
    'winner_label': str(winner['label']),
    'r_overall':    float(winner['r_overall']),
    'r_male':       float(winner['r_male']),
    'r_female':     float(winner['r_female']),
    'mae':          float(winner['mae']),
    'beats_paper':  bool(winner['r_overall'] >= 0.65),
}

# Map winner label to model file + feature extractors needed for the demo
if 'Concat' in winner['label']:
    winner_info['model_file'] = 'svr_concat_tuned.joblib'
    winner_info['feature_extractors'] = ['VGG-Face', 'Facenet512']
elif 'concat' in winner['label']:
    winner_info['model_file'] = 'svr_wider_concat.joblib'
    winner_info['feature_extractors'] = ['VGG-Face', 'Facenet512']
elif 'Ensemble' in winner['label'] or 'ensemble' in winner['label']:
    winner_info['model_file'] = 'ensemble (uses both baseline models)'
    winner_info['feature_extractors'] = ['VGG-Face', 'Facenet512']
elif 'VGG-Face' in winner['label']:
    winner_info['model_file'] = 'svr_vgg_tuned.joblib'
    winner_info['feature_extractors'] = ['VGG-Face']
elif 'FaceNet' in winner['label']:
    winner_info['model_file'] = 'svr_facenet_tuned.joblib'
    winner_info['feature_extractors'] = ['Facenet512']
else:
    winner_info['model_file'] = 'unknown'
    winner_info['feature_extractors'] = []

with open(f'{MODELS_DIR}/winner_info.json', 'w') as f:
    json.dump(winner_info, f, indent=2)

print('Winner info saved:')
print(json.dumps(winner_info, indent=2))

Winner info saved:
{
  "winner_label": "B: Ensemble (avg VGG + FaceNet)",
  "r_overall": 0.6468831999158444,
  "r_male": 0.6555200036468499,
  "r_female": 0.6427199522095624,
  "mae": 5.113365059616409,
  "beats_paper": false,
  "model_file": "ensemble (uses both baseline models)",
  "feature_extractors": [
    "VGG-Face",
    "Facenet512"
  ]
}


## What to come back with

The full `final_df` table from Cell 11. We'll discuss:
- Did we beat 0.65?
- Which technique helped most?
- Any surprising findings worth documenting in the write-up?

## Next: Phase 5 (demo)

We'll write a Streamlit app that:
1. Lets the user upload a face image OR capture from webcam
2. Runs the winning pipeline (feature extraction → SVR prediction)
3. Shows predicted BMI with an honest uncertainty range

## And after: Phase 6 (write-up)

10 pages on what we did, what we found, what worked, what didn't. The honest narrative version, not a summary of the paper.